In [1]:
pip install pypdf

Note: you may need to restart the kernel to use updated packages.


In [9]:
pip install langchain langchain-community langchain-text-splitters

Note: you may need to restart the kernel to use updated packages.


In [4]:
pip install sentence-transformers

   ---------------------------------------- 0.0/740.6 kB ? eta -:--:--
   ---------------------------------------- 0.0/740.6 kB ? eta -:--:--
   ---------------------------- ----------- 524.3/740.6 kB 1.5 MB/s eta 0:00:01
   ---------------------------------------- 740.6/740.6 kB 1.6 MB/s  0:00:00
   ---------------------------------------- 0.0/12.3 MB ? eta -:--:--
    --------------------------------------- 0.3/12.3 MB ? eta -:--:--
   -- ------------------------------------- 0.8/12.3 MB 2.0 MB/s eta 0:00:06
   --- ------------------------------------ 1.0/12.3 MB 2.0 MB/s eta 0:00:06
   ----- ---------------------------------- 1.6/12.3 MB 2.0 MB/s eta 0:00:06
   ----- ---------------------------------- 1.8/12.3 MB 1.9 MB/s eta 0:00:06
   ------ --------------------------------- 2.1/12.3 MB 1.8 MB/s eta 0:00:06
   ------ --------------------------------- 2.1/12.3 MB 1.8 MB/s eta 0:00:06
   ------ --------------------------------- 2.1/12.3 MB 1.8 MB/s eta 0:00:06
   ------- -----------

ERROR: Could not install packages due to an OSError: [Errno 2] No such file or directory: 'C:\\Users\\DELL UZER\\ana\\Lib\\site-packages\\regex-2026.9.10.dist-info\\INSTALLERrc5ra60t.tmp'



In [10]:
pip install faiss-cpu

Note: you may need to restart the kernel to use updated packages.


In [11]:
pip install streamlit

Note: you may need to restart the kernel to use updated packages.


In [12]:
pip install pandas numpy scikit-learn

Note: you may need to restart the kernel to use updated packages.


In [16]:
import pypdf
import langchain
import langchain_community
import sentence_transformers
import faiss
import streamlit
import pandas
import numpy
import sklearn

In [3]:
pip install reportlab

Note: you may need to restart the kernel to use updated packages.


In [5]:
from reportlab.pdfgen import canvas


In [6]:
from reportlab.pdfgen import canvas
pdf_path = "documents/sample.pdf"
c = canvas.Canvas(pdf_path)
c.setFont("Helvetica-Bold", 16)
c.drawString(50, 800, "Employee Leave Policy")
c.setFont("Helvetica", 11)
text = [
    "Employee Leave Policy",
    "",
    "1. Annual Leave",
    "Employees are entitled to 12 days of annual leave per year.",
    "",
    "2. Leave Request",
    "Employees must submit leave requests through the HR portal.",
    "",
    "3. Medical Leave",
    "Medical leave requires a valid medical certificate.",
    "",
    "4. Leave Approval",
    "The manager must approve leave requests before the leave begins.",
    "",
    "5. Carry Forward",
    "Employees can carry forward up to 5 unused leave days",
    "to the following year."
]
y = 770
for line in text:
    c.drawString(50, y, line)
    y -= 25
c.save()
print("sample.pdf created successfully!")

sample.pdf created successfully!


In [7]:
from pypdf import PdfReader
pdf_path = "documents/sample.pdf"
reader = PdfReader(pdf_path)
print("Number of pages:", len(reader.pages))

Number of pages: 1


In [9]:
text = ""

for page in reader.pages:
    page_text = page.extract_text()
    
    if page_text:
        text += page_text + "\n"

print(text)

Employee Leave Policy
Employee Leave Policy
1. Annual Leave
Employees are entitled to 12 days of annual leave per year.
2. Leave Request
Employees must submit leave requests through the HR portal.
3. Medical Leave
Medical leave requires a valid medical certificate.
4. Leave Approval
The manager must approve leave requests before the leave begins.
5. Carry Forward
Employees can carry forward up to 5 unused leave days
to the following year.




In [12]:
from langchain_text_splitters import RecursiveCharacterTextSplitter


In [14]:
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=300,
    chunk_overlap=50
)


In [15]:
chunks = text_splitter.split_text(text)
print("Number of chunks:", len(chunks))

Number of chunks: 2


In [16]:
for i, chunk in enumerate(chunks):
    print(f"\n--- Chunk {i+1} ---")
    print(chunk)


--- Chunk 1 ---
Employee Leave Policy
Employee Leave Policy
1. Annual Leave
Employees are entitled to 12 days of annual leave per year.
2. Leave Request
Employees must submit leave requests through the HR portal.
3. Medical Leave
Medical leave requires a valid medical certificate.
4. Leave Approval

--- Chunk 2 ---
4. Leave Approval
The manager must approve leave requests before the leave begins.
5. Carry Forward
Employees can carry forward up to 5 unused leave days
to the following year.


In [17]:
for i, chunk in enumerate(chunks):
    print(f"Chunk {i+1}: {len(chunk)} characters")

Chunk 1: 283 characters
Chunk 2: 176 characters


In [19]:
from sentence_transformers import SentenceTransformer

In [23]:
embedding_model = SentenceTransformer("all-MiniLM-L6-v2")


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

In [22]:
chunk_embeddings = embedding_model.encode(chunks)
print("Number of embeddings:", len(chunk_embeddings))

Number of embeddings: 2


In [25]:
import faiss

In [26]:
embedding_dimension = chunk_embeddings.shape[1]
print("Embedding dimension:", embedding_dimension)

Embedding dimension: 384


In [34]:
index = faiss.IndexFlatL2(chunk_embeddings.shape[1])
index.add(chunk_embeddings.astype("float32"))
print("Number of vectors in FAISS:", index.ntotal)

Number of vectors in FAISS: 2


In [35]:
query = "How many days of annual leave are employees entitled to?"
print("Question:", query)

Question: How many days of annual leave are employees entitled to?


In [36]:
query_embedding = embedding_model.encode([query])
print("Query embedding shape:", query_embedding.shape)

Query embedding shape: (1, 384)


In [37]:
distances, indices = index.search(
    query_embedding.astype("float32"),
    k=2
)
print("Distances:", distances)
print("Indices:", indices)

Distances: [[0.67504305 0.8726164 ]]
Indices: [[0 1]]


In [38]:
for i in indices[0]:
    print("\n--- Retrieved Chunk ---")
    print(chunks[i])


--- Retrieved Chunk ---
Employee Leave Policy
Employee Leave Policy
1. Annual Leave
Employees are entitled to 12 days of annual leave per year.
2. Leave Request
Employees must submit leave requests through the HR portal.
3. Medical Leave
Medical leave requires a valid medical certificate.
4. Leave Approval

--- Retrieved Chunk ---
4. Leave Approval
The manager must approve leave requests before the leave begins.
5. Carry Forward
Employees can carry forward up to 5 unused leave days
to the following year.


In [40]:
import os
os.makedirs("vector_db", exist_ok=True)
faiss.write_index(index, "vector_db/faiss_index.bin")

In [41]:
import pickle
with open("vector_db/chunks.pkl", "wb") as f:
    pickle.dump(chunks, f)

In [42]:
print(os.listdir("vector_db"))

['chunks.pkl', 'faiss_index.bin']


In [43]:
pip install ollama

Note: you may need to restart the kernel to use updated packages.


In [3]:
import ollama

In [4]:
response = ollama.chat(
    model="llama3.2",
    messages=[
        {
            "role": "user",
            "content": "what is artificial intelligence?"
        }
    ]
)
print(response["message"]["content"])

Artificial intelligence (AI) refers to the development of computer systems that can perform tasks that typically require human intelligence, such as learning, problem-solving, and decision-making. The term "artificial" means that the system is not human, while "intelligence" refers to the ability to perform tasks that would normally require human intelligence.

AI systems use algorithms and data to make decisions, classify patterns, and recognize objects. They can also be trained to perform specific tasks, such as recognizing speech, understanding natural language, and recognizing images.

There are several types of AI, including:

1. **Narrow or weak AI**: This type of AI is designed to perform a specific task, such as facial recognition or language translation.
2. **General or strong AI**: This type of AI is designed to perform any intellectual task that a human can, and is still a subject of ongoing research and development.
3. **Superintelligence**: This type of AI is significantly

In [12]:
import pickle
with open("vector_db/chunks.pkl", "rb") as f:
    chunks = pickle.load(f)
print("Number of chunks:", len(chunks))

Number of chunks: 2


In [13]:
import faiss
index = faiss.read_index("vector_db/faiss_index.bin")
print("Number of FAISS vectors:", index.ntotal)

Number of FAISS vectors: 2


In [14]:
from sentence_transformers import SentenceTransformer
embedding_model = SentenceTransformer("all-MiniLM-L6-v2")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

In [16]:
query = "How many days of annual leave are employees entitled to?"
query_embedding = embedding_model.encode([query])
distances, indices = index.search(
    query_embedding.astype("float32"),
    k=1
)
retrieved_chunk = chunks[indices[0][0]]
print("Question:", query)
print("\nRetrieved context:")
print(retrieved_chunk)

Question: How many days of annual leave are employees entitled to?

Retrieved context:
Employee Leave Policy
Employee Leave Policy
1. Annual Leave
Employees are entitled to 12 days of annual leave per year.
2. Leave Request
Employees must submit leave requests through the HR portal.
3. Medical Leave
Medical leave requires a valid medical certificate.
4. Leave Approval


In [17]:
prompt = f"""
You are a document question-answering assistant.

Answer the question using ONLY the information in the context below.

If the answer is not found in the context, say:
"I could not find the answer in the provided document."

Context:
{retrieved_chunk}

Question:
{query}

Answer:
"""
response = ollama.chat(
    model="llama3.2",
    messages=[
        {
            "role": "user",
            "content": prompt
        }
    ]
)
answer = response["message"]["content"]
print("Answer:")
print(answer)

Answer:
Employees are entitled to 12 days of annual leave per year.


In [19]:
test_questions = [
    "What is required for medical leave?",
    "How many unused leave days can employees carry forward?",
    "Who must approve leave requests?"
]
for question in test_questions:
    query_embedding = embedding_model.encode([question])

    distances, indices = index.search(
        query_embedding.astype("float32"),
        k=1
    )
    retrieved_chunk = chunks[indices[0][0]]
    prompt = f"""
You are a document question-answering assistant.

Answer the question using ONLY the information in the context.

If the answer is not present in the context, say:
"I could not find the answer in the provided document."

Context:
{retrieved_chunk}

Question:
{question}

Answer:
"""

    response = ollama.chat(
        model="llama3.2",
        messages=[
            {
                "role": "user",
                "content": prompt
            }
        ]
    )

    print("Question:", question)
    print("Answer:", response["message"]["content"])

Question: What is required for medical leave?
Answer: A valid medical certificate is required for medical leave.
Question: How many unused leave days can employees carry forward?
Answer: 5
Question: Who must approve leave requests?
Answer: The manager must approve leave requests.


In [20]:
question = "What is the employee's monthly salary?"
query_embedding = embedding_model.encode([question])
distances, indices = index.search(
    query_embedding.astype("float32"),
    k=1
)
retrieved_chunk = chunks[indices[0][0]]
prompt = f"""
You are a document question-answering assistant.

Answer the question using ONLY the information in the context.

If the answer is not present in the context, say:
"I could not find the answer in the provided document."

Do not guess or use outside knowledge.

Context:
{retrieved_chunk}

Question:
{question}

Answer:
"""

response = ollama.chat(
    model="llama3.2",
    messages=[
        {
            "role": "user",
            "content": prompt
        }
    ]
)

print("Question:", question)
print("Answer:", response["message"]["content"])

Question: What is the employee's monthly salary?
Answer: I could not find the answer in the provided document.


In [21]:
from langchain_community.document_loaders import PyPDFLoader
pdf_path = "documents/sample.pdf"
loader = PyPDFLoader(pdf_path)
documents = loader.load()
print("Number of pages:", len(documents))
print("\nFirst page metadata:")
print(documents[0].metadata)

C:\Users\DELL UZER\AppData\Local\Temp\ipykernel_18996\1172510281.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader


Number of pages: 1

First page metadata:
{'producer': 'ReportLab PDF Library - (opensource)', 'creator': 'anonymous', 'creationdate': '2026-09-23T18:28:37+05:00', 'author': 'anonymous', 'keywords': '', 'moddate': '2026-09-23T18:28:37+05:00', 'subject': 'unspecified', 'title': 'untitled', 'trapped': '/False', 'source': 'documents/sample.pdf', 'total_pages': 1, 'page': 0, 'page_label': '1'}


In [22]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=300,
    chunk_overlap=50
)
document_chunks = text_splitter.split_documents(documents)
print("Number of chunks:", len(document_chunks))
for i, chunk in enumerate(document_chunks):
    print(f"\n--- Chunk {i+1} ---")
    print(chunk.page_content)
    print("Source:", chunk.metadata["source"])
    print("Page:", chunk.metadata["page_label"])

Number of chunks: 2

--- Chunk 1 ---
Employee Leave Policy
Employee Leave Policy
1. Annual Leave
Employees are entitled to 12 days of annual leave per year.
2. Leave Request
Employees must submit leave requests through the HR portal.
3. Medical Leave
Medical leave requires a valid medical certificate.
4. Leave Approval
Source: documents/sample.pdf
Page: 1

--- Chunk 2 ---
4. Leave Approval
The manager must approve leave requests before the leave begins.
5. Carry Forward
Employees can carry forward up to 5 unused leave days
to the following year.
Source: documents/sample.pdf
Page: 1


In [23]:
document_texts = [doc.page_content for doc in document_chunks]
document_embeddings = embedding_model.encode(document_texts)
print("Number of embeddings:", len(document_embeddings))
print("Embedding shape:", document_embeddings.shape)

Number of embeddings: 2
Embedding shape: (2, 384)


In [24]:
import faiss
document_index = faiss.IndexFlatL2(document_embeddings.shape[1])
document_index.add(
    document_embeddings.astype("float32")
)
print("Number of vectors:", document_index.ntotal)

Number of vectors: 2


In [25]:
import pickle
import faiss
import os
os.makedirs("vector_db", exist_ok=True)
with open("vector_db/document_chunks.pkl", "wb") as f:
    pickle.dump(document_chunks, f)

faiss.write_index(
    document_index,
    "vector_db/document_faiss_index.bin"
)
print("Metadata-aware RAG files saved successfully!")

Metadata-aware RAG files saved successfully!


In [26]:
import os
print(os.listdir("vector_db"))

['chunks.pkl', 'document_chunks.pkl', 'document_faiss_index.bin', 'faiss_index.bin']


In [27]:
query = "How many days of annual leave are employees entitled to?"
query_embedding = embedding_model.encode([query])
distances, indices = document_index.search(
    query_embedding.astype("float32"),
    k=1
)
retrieved_document = document_chunks[indices[0][0]]
print("Question:", query)
print(retrieved_document.page_content)
print("Source:", retrieved_document.metadata["source"])
print("Page:", retrieved_document.metadata["page_label"])

Question: How many days of annual leave are employees entitled to?
Employee Leave Policy
Employee Leave Policy
1. Annual Leave
Employees are entitled to 12 days of annual leave per year.
2. Leave Request
Employees must submit leave requests through the HR portal.
3. Medical Leave
Medical leave requires a valid medical certificate.
4. Leave Approval
Source: documents/sample.pdf
Page: 1


In [28]:
query = "How many days of annual leave are employees entitled to?"
query_embedding = embedding_model.encode([query])
distances, indices = document_index.search(
    query_embedding.astype("float32"),
    k=1
)
retrieved_document = document_chunks[indices[0][0]]
context = retrieved_document.page_content
source = retrieved_document.metadata["source"]
page = retrieved_document.metadata["page_label"]
prompt = f"""
You are a document question-answering assistant.

Answer the question using ONLY the information provided in the context.

If the answer is not present in the context, say:
"I could not find the answer in the provided document."

Context:
{context}

Question:
{query}

Answer:
"""

response = ollama.chat(
    model="llama3.2",
    messages=[
        {
            "role": "user",
            "content": prompt
        }
    ]
)
answer = response["message"]["content"]
print("Question:", query)
print(answer)
print(source)
print("Page:")
print(page)

Question: How many days of annual leave are employees entitled to?
Employees are entitled to 12 days of annual leave per year.
documents/sample.pdf
Page:
1


In [29]:
def ask_rag(question):
    query_embedding = embedding_model.encode([question])

    distances, indices = document_index.search(
        query_embedding.astype("float32"),
        k=1
    )

    retrieved_document = document_chunks[indices[0][0]]

    context = retrieved_document.page_content
    source = retrieved_document.metadata["source"]
    page = retrieved_document.metadata["page_label"]

    prompt = f"""
You are a document question-answering assistant.

Answer the question using ONLY the information provided in the context.

If the answer is not present in the context, say:
"I could not find the answer in the provided document."

Context:
{context}

Question:
{question}

Answer:
"""

    response = ollama.chat(
        model="llama3.2",
        messages=[
            {
                "role": "user",
                "content": prompt
            }
        ]
    )

    answer = response["message"]["content"]

    return answer, source, page

In [30]:
answer, source, page = ask_rag(
    "What is required for medical leave?"
)
print("Answer:", answer)
print("Source:", source)
print("Page:", page)

Answer: A valid medical certificate is required for medical leave.
Source: documents/sample.pdf
Page: 1


In [32]:
test_questions = [
    "How many days of annual leave are employees entitled to?",
    "What is required for medical leave?",
    "Who must approve leave requests?",
    "What is the employee's monthly salary?"
]
for question in test_questions:
    answer, source, page = ask_rag(question)

    print("Question:", question)
    print("Answer:", answer)
    print("Source:", source)
    print("Page:", page)

Question: How many days of annual leave are employees entitled to?
Answer: Employees are entitled to 12 days of annual leave per year.
Source: documents/sample.pdf
Page: 1
Question: What is required for medical leave?
Answer: Medical leave requires a valid medical certificate.
Source: documents/sample.pdf
Page: 1
Question: Who must approve leave requests?
Answer: The manager must approve leave requests.
Source: documents/sample.pdf
Page: 1
Question: What is the employee's monthly salary?
Answer: I could not find the answer in the provided document.
Source: documents/sample.pdf
Page: 1
